# MOSAIC Categorical Values Detector
**Detection-only.** Read-only against source data. Every finding requires steward review before any action.

## Two operating modes

This detector works differently from the other detectors in the suite because there is no universal "correct" canonical set — it is per-column and per-product. The detector operates in two modes simultaneously:

**Inventory mode** (no canonical set registered): Discovers and profiles categorical columns that have no registered canonical value set. Surfaces cardinality, distinct values, and AI-classified domain so stewards can register the set.

**Compliance mode** (canonical set provided): Measures compliance of column values against a registered canonical set, flagging out-of-set values, alias patterns, cardinality drift, and raw values reaching Gold without mapping.

## Providing a canonical set

Two methods (configured via widgets):

**Method A — Inline (small, stable sets):** Set `canonical_inline` widget to a JSON object mapping column-name patterns to comma-separated canonical values. Example: `{"gender": "M,F,U,O,X", "claim_status": "ACTIVE,DENIED,PAID,PENDING,VOIDED"}`.

**Method B — Lookup table (preferred per procedure):** Set `canonical_catalog`, `canonical_schema`, `canonical_table` widgets to point to a Delta table with columns `domain STRING, canonical_value STRING`. The detector joins against it at runtime.

| Cell | What it does | Procedure §§ |
|---|---|---|
| 1 - Overview | This document | — |
| 2 - Rule Reference | Full rule definitions, standardization options, decision workflow | All |
| 3 - Config | Parameters / widgets | — |
| 4 - Constants | Tags, severity, column name heuristics, standardization rules | §Inventory, §Standardization |
| 5 - Discovery | information_schema scan; identify low-cardinality STRING candidates | §Inventory |
| 6 - Profiler | Per-table: distinct_count, null_count, total, distinct value samples | §Inventory, §Validation |
| 7 - AI Classifier | Confirm categorical columns; classify domain; detect aliases and out-of-set | §Standardization, §Validation |
| 8 - Rule Engine | Evaluate compliance, inventory, drift, alias patterns | All |
| 9 - Write Results | Append findings to results Delta table | — |
| 10 - Summary | Blocking (Gold certification) → by rule → by domain → steward work queue | §Validation, §Enforcement |

> **Hard constraint:** UPDATE, MERGE, DELETE, CTAS on source tables are never executed.


# MOSAIC Categorical Values Detector — Rule Reference

**Detection-only.** No source data is modified.

---

## §Inventory
| Rule tag | What it detects | Required action |
|---|---|---|
| `UNREGISTERED_CATEGORICAL` | Column looks categorical but has no canonical value set registered | Steward registers canonical set in lookup table or inline mapping artifact |
| `RAW_VALUES_IN_GOLD` | Gold/certified layer column has no canonical mapping — raw source values are reaching certified models | Register canonical set and apply mapping in ETL; certification blocker |

## §Standardization
| Rule tag | What it detects | Required action |
|---|---|---|
| `INLINE_MAPPING_RISK` | Column uses inline mapping (small set) but cardinality or instability suggests a lookup table is needed | Migrate to lookup table with source_value → canonical_value mapping per §Standardization |
| `MIXED_CASE_CATEGORICAL` | Same logical value stored in multiple cases (male/Male/MALE, active/Active/ACTIVE) | Normalize case in Silver ETL; register one canonical form in the set |

## §Validation
| Rule tag | What it detects | Required action |
|---|---|---|
| `OUT_OF_SET_VALUE` | One or more values not in the approved canonical set | NULL or quarantine per §Validation; do not pass raw strings to Gold |
| `HIGH_OUT_OF_SET_RATE` | > threshold % of non-null values not in canonical set | Escalate to steward; block certification per §Enforcement |
| `UNMAPPED_VALUES` | Inventory of distinct values not in canonical set | Feed into alias table / mapping PR; steward review per §Change-control |
| `UNDOCUMENTED_ALIAS` | AI detected a value that appears to be an alias of a canonical value (e.g. Male for M, Active for ACTIVE) | Add to alias table via PR/review per §Change-control |
| `HIGH_NULL_RATE` | > threshold % of values are NULL in a categorical column | Investigate source; may indicate upstream mapping failure |

## §Change control
| Rule tag | What it detects | Required action |
|---|---|---|
| `CARDINALITY_DRIFT` | Distinct count exceeds expected_cardinality for this column | Investigate new value; if new canonical category → product owner sign-off per §Change-control |
| `NEW_VALUE_APPEARED` | A distinct value is present that was not in the canonical set and was not present in the previous run | Alert steward; requires PR/review before alias is approved |

---

## Decision workflow
1. **Inventory mode:** UNREGISTERED_CATEGORICAL findings go to steward → steward registers canonical set → re-run in compliance mode.
2. **Compliance mode:** OUT_OF_SET_VALUE and UNMAPPED_VALUES go to engineer → add to alias table via PR → steward approves → re-run to confirm.
3. CARDINALITY_DRIFT and NEW_VALUE_APPEARED require product owner sign-off before new canonical value is added.
4. RAW_VALUES_IN_GOLD is a certification blocker until mapping is registered.

## Enforcement
Uncertified category drift in Gold layers blocks certification until mappings are remediated or risk-accepted in writing.


In [0]:
import re, json, uuid
from datetime import datetime
from typing import Any, Dict, List, Tuple, Set, Optional
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)

CFG: Dict[str, Any] = {
    # Source
    "catalog":      _w("catalog",      "data_classification_source"),
    "schema":       _w("schema",       "categorical_values_detector"),
    "table_filter": _w("table_filter", ""),
    "skip_views":   _w("skip_views",   "true").strip().lower() == "true",

    # Layer -- RAW_VALUES_IN_GOLD only fires in Gold/certified layers
    "layer": _w("layer", "Unknown"),   # Bronze | Silver | Gold | Unknown

    # Sampling
    "sample_size": int(_w("sample_size", 100_000)),
    "seed":        int(_w("seed",        42)),

    # Candidate detection thresholds
    # A STRING column is a categorical candidate when:
    #   distinct_count / total <= cardinality_ratio AND distinct_count <= max_cardinality
    #   AND avg_length <= max_avg_length AND total >= min_rows
    "cardinality_ratio":  float(_w("cardinality_ratio",  "0.05")),  # <=5% distinct/total
    "max_cardinality":    int(_w("max_cardinality",       "100")),   # absolute cap
    "max_avg_length":     float(_w("max_avg_length",      "50.0")),  # excludes narratives
    "min_rows":           int(_w("min_rows",              "10")),    # need enough data

    # Compliance thresholds
    "out_of_set_threshold": float(_w("out_of_set_threshold", "0.0")),  # any OOS value fires
    "high_oos_threshold":   float(_w("high_oos_threshold",   "5.0")),  # HIGH_OUT_OF_SET_RATE
    "null_rate_threshold":  float(_w("null_rate_threshold",  "10.0")), # HIGH_NULL_RATE

    # Canonical set -- Method A: inline JSON mapping
    # Example: {"gender": "M,F,U,O,X", "claim_status": "ACTIVE,DENIED,PAID,PENDING"}
    # Column name must contain the key as a substring (case-insensitive)
    "canonical_inline": _w("canonical_inline", "{}"),

    # Canonical set -- Method B: lookup table (preferred)
    # Table must have columns: domain STRING, canonical_value STRING
    # domain is matched against column name (substring, case-insensitive)
    "canonical_catalog": _w("canonical_catalog", ""),
    "canonical_schema":  _w("canonical_schema",  ""),
    "canonical_table":   _w("canonical_table",   ""),

    # Expected cardinality per domain (for CARDINALITY_DRIFT)
    # Example: {"gender": "5", "claim_status": "8"}
    "expected_cardinality": _w("expected_cardinality", "{}"),

    # AI
    "ai_model":  _w("ai_model",  "databricks-meta-llama-3-3-70b-instruct"),
    "enable_ai": _w("enable_ai", "true").strip().lower() == "true",

    # Results
    "out_catalog": _w("out_catalog", "data_classification_target"),
    "out_schema":  _w("out_schema",  "data_classification_output"),
    "out_table":   _w("out_table",   "categorical_findings"),
}

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()

# Parse inline canonical sets
try:
    CANONICAL_INLINE: Dict[str, Set[str]] = {
        k: {v.strip().upper() for v in vs.split(",") if v.strip()}
        for k, vs in json.loads(CFG["canonical_inline"]).items()
    }
except Exception:
    CANONICAL_INLINE = {}

# Parse expected cardinality overrides
try:
    EXPECTED_CARDINALITY: Dict[str, int] = {
        k: int(v) for k, v in json.loads(CFG["expected_cardinality"]).items()
    }
except Exception:
    EXPECTED_CARDINALITY = {}

print(f"Run            : {RUN_ID}")
print(f"Scope          : {CFG['catalog']}.{CFG['schema']}")
print(f"Layer          : {CFG['layer']}")
print(f"AI             : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off (heuristic-only)'}")
print(f"Inline sets    : {list(CANONICAL_INLINE.keys()) or 'none'}")
print(f"Lookup table   : {CFG['canonical_catalog']}.{CFG['canonical_schema']}.{CFG['canonical_table'] or '(not configured)'}")
print(f"Results        : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# HOW TO ADD A NEW RULE:
#   1. Add tag to TAGS with procedure section reference
#   2. Add severity to SEVERITY
#   3. Add standardization options to STANDARDIZATION_RULES
#   4. Add detection logic in _check_categorical() in Cell 6
# ---------------------------------------------------------------------------

TAGS = {
    # §Inventory
    "UNREGISTERED_CATEGORICAL":  "§Inventory",
    "RAW_VALUES_IN_GOLD":        "§Inventory",
    # §Standardization
    "INLINE_MAPPING_RISK":       "§Standardization",
    "MIXED_CASE_CATEGORICAL":    "§Standardization",
    # §Validation
    "OUT_OF_SET_VALUE":          "§Validation",
    "HIGH_OUT_OF_SET_RATE":      "§Validation",
    "UNMAPPED_VALUES":           "§Validation",
    "UNDOCUMENTED_ALIAS":        "§Validation",
    "HIGH_NULL_RATE":            "§Validation",
    # §Change control
    "CARDINALITY_DRIFT":         "§Change-control",
    "NEW_VALUE_APPEARED":        "§Change-control",
}

SEVERITY: Dict[str, int] = {
    "RAW_VALUES_IN_GOLD":       10,
    "OUT_OF_SET_VALUE":         10,
    "HIGH_OUT_OF_SET_RATE":      9,
    "UNMAPPED_VALUES":           8,
    "CARDINALITY_DRIFT":         8,
    "NEW_VALUE_APPEARED":        8,
    "UNDOCUMENTED_ALIAS":        7,
    "MIXED_CASE_CATEGORICAL":    6,
    "HIGH_NULL_RATE":            5,
    "UNREGISTERED_CATEGORICAL":  5,
    "INLINE_MAPPING_RISK":       4,
}

# ── Column name heuristics ───────────────────────────────────────────────────
# Strong signal: column name clearly indicates a categorical/status/type field
RE_CATEGORICAL_NAME = re.compile(
    r"(status|type|gender|sex|race|ethnicity|marital|flag|ind|indicator|"
    r"category|class|tier|level|priority|code|cd|_cd$|_type$|_status$|"
    r"source|method|mode|channel|reason|outcome|result_type|"
    r"enrollment|coverage|plan_type|claim_type|service_type|"
    r"diagnosis_type|procedure_type|provider_type|facility_type|"
    r"language|preferred_language|contact_type|address_type|"
    r"relationship|member_type|patient_type|encounter_type)",
    re.I
)
# Exclude columns that match categorical name patterns but are not categoricals
RE_CATEGORICAL_EXCLUDE = re.compile(
    r"(_id$|_key$|_num$|_number$|_nbr$|_date$|_ts$|_timestamp$|"
    r"_amount$|_amt$|_count$|_cnt$|_rate$|_pct$|_percent$|"
    r"_description$|_desc$|_note$|_comment$|_text$|_narrative$|"
    r"source_value$|source_code$|raw_value$)",
    re.I
)

# ── Domain taxonomy for AI classification ────────────────────────────────────
CATEGORICAL_DOMAINS = {
    "gender_sex",           # M/F/U/O/X or Male/Female/Unknown
    "race_ethnicity",       # OMB/CDCREC codes or display values
    "claim_status",         # ACTIVE/DENIED/PAID/PENDING/VOIDED/REVERSED
    "enrollment_status",    # ACTIVE/INACTIVE/TERMINATED/PENDING
    "coverage_type",        # plan type, coverage tier
    "diagnosis_category",   # ICD category groupings
    "service_type",         # professional/facility/pharmacy/dental
    "provider_type",        # PCP/specialist/hospital/etc
    "address_type",         # mailing/billing/home/work
    "contact_type",         # phone/email/fax/text
    "language",             # preferred language codes
    "yes_no_flag",          # Y/N, 1/0, true/false, yes/no
    "payer_type",           # commercial/medicare/medicaid/self-pay
    "relationship",         # self/spouse/child/dependent
    "encounter_type",       # inpatient/outpatient/ER/telehealth
    "discharge_status",     # routine/expired/AMA/transfer
    "other",                # categorical but domain not recognized
}

STANDARDIZATION_RULES: Dict[str, list] = {

    "UNREGISTERED_CATEGORICAL": [
        {"option_key": "register_canonical_set",
         "label":      "Register canonical value set in lookup table or inline mapping",
         "sql":        "-- INSERT INTO <canonical_table> (domain, canonical_value) VALUES ...",
         "notes":      "Catalog every categorical column in the data dictionary with expected "
                       "cardinality and steward owner (§Inventory). "
                       "Preferred: lookup table with source_value → canonical_value mapping (§Standardization). "
                       "Inline mapping only for small, stable, documented sets (§Standardization)."},
    ],

    "RAW_VALUES_IN_GOLD": [
        {"option_key": "register_and_map",
         "label":      "Register canonical set and apply ETL mapping before Gold promotion",
         "sql":        "-- SILVER → GOLD: JOIN source ON source.source_value = col "
                       "→ emit source.canonical_value. "
                       "-- Or CASE WHEN col IN (<set>) THEN col ELSE NULL END",
         "notes":      "Do not pass raw source strings to Gold/certified models when a canonical "
                       "set exists (§Standardization). "
                       "Uncertified category drift in Gold blocks certification (§Enforcement)."},
    ],

    "INLINE_MAPPING_RISK": [
        {"option_key": "migrate_to_lookup_table",
         "label":      "Migrate inline CASE mapping to a lookup table",
         "sql":        "-- CREATE TABLE <canonical_table> (domain STRING, source_value STRING, "
                       "canonical_value STRING, effective_from DATE, effective_to DATE)",
         "notes":      "Inline mapping acceptable only when cardinality is small, stable, and "
                       "documented in the mapping artifact (§Standardization). "
                       "Use lookup table when cardinality > threshold or values change frequently."},
    ],

    "MIXED_CASE_CATEGORICAL": [
        {"option_key": "normalize_case",
         "label":      "Normalize case to canonical form in Silver ETL",
         "sql":        "-- SILVER: UPPER(TRIM(col)) AS col  (or match canonical set casing)",
         "notes":      "Apply case normalization before canonical set join (§Standardization). "
                       "Register one canonical form per logical value in the lookup table."},
    ],

    "OUT_OF_SET_VALUE": [
        {"option_key": "null_oos",
         "label":      "Set out-of-set values to NULL in Silver transform",
         "sql":        "-- SILVER: CASE WHEN col IN (<canonical_set>) THEN col ELSE NULL END AS col",
         "notes":      "DQ rules must fail or flag rows where post-transform value not in "
                       "approved canonical set (§Validation). "
                       "Do not pass raw strings to Gold when canonical set exists (§Standardization)."},
        {"option_key": "quarantine_oos",
         "label":      "Route out-of-set rows to quarantine table",
         "sql":        "-- Route rows WHERE col NOT IN (<canonical_set>) to quarantine.",
         "notes":      "Use when steward needs to review out-of-set values before nulling. "
                       "Feed quarantine into alias table review (§Change-control)."},
        {"option_key": "add_to_alias_table",
         "label":      "Add out-of-set value as alias in alias table (requires PR/review)",
         "sql":        "-- INSERT INTO alias_table (domain, source_value, canonical_value) VALUES ...",
         "notes":      "Adding aliases requires PR/review (§Change-control). "
                       "Do not add new canonical values without steward approval (§Standardization)."},
    ],

    "HIGH_OUT_OF_SET_RATE": [
        {"option_key": "escalate_and_quarantine",
         "label":      "Escalate to steward and quarantine all out-of-set rows",
         "sql":        "-- Route all out-of-set rows to quarantine. Block certification.",
         "notes":      "Uncertified category drift in Gold layers blocks certification (§Enforcement). "
                       "Monitor alias table for unmapped values weekly during onboarding, "
                       "daily for production-critical domains (§Validation)."},
    ],

    "UNMAPPED_VALUES": [
        {"option_key": "map_via_alias_table",
         "label":      "Add unmapped values to alias table via PR/review",
         "sql":        "-- For each unmapped value: INSERT INTO alias_table "
                       "(domain, source_value, canonical_value) VALUES (...)",
         "notes":      "Adding aliases requires PR/review (§Change-control). "
                       "Monitor alias table weekly during onboarding, daily for production-critical (§Validation)."},
    ],

    "UNDOCUMENTED_ALIAS": [
        {"option_key": "register_alias",
         "label":      "Register detected alias in alias table via PR/review",
         "sql":        "-- INSERT INTO alias_table (domain, source_value, canonical_value) "
                       "VALUES ('<alias>', '<canonical>')",
         "notes":      "Adding aliases requires PR/review per §Change-control. "
                       "AI detected this value as a likely alias -- confirm before registering."},
    ],

    "HIGH_NULL_RATE": [
        {"option_key": "investigate_source",
         "label":      "Investigate source for upstream mapping failure or data loss",
         "sql":        "-- No transform. Investigate source system and pipeline mapping.",
         "notes":      "High null rate in categorical column may indicate upstream mapping failure "
                       "or wrong join (§Validation)."},
    ],

    "CARDINALITY_DRIFT": [
        {"option_key": "investigate_new_value",
         "label":      "Investigate new value; product owner sign-off if new canonical category",
         "sql":        "-- SELECT DISTINCT col FROM tbl WHERE col NOT IN (<known_set>)",
         "notes":      "Adding new canonical categories requires product owner sign-off (§Change-control). "
                       "Do not add new canonical values without steward approval and dictionary update."},
    ],

    "NEW_VALUE_APPEARED": [
        {"option_key": "alert_and_review",
         "label":      "Alert steward; review via PR before adding to alias or canonical set",
         "sql":        "-- Flag new value for steward review. Block if Gold/certified layer.",
         "notes":      "Adding aliases requires PR/review (§Change-control). "
                       "Adding new canonical categories requires product owner sign-off."},
    ],

    "MIXED_CASE_CATEGORICAL": [
        {"option_key": "normalize_case_and_register",
         "label":      "Normalize case and register canonical form in lookup table",
         "sql":        "-- SILVER: UPPER(TRIM(col)) AS col  -- then join alias table",
         "notes":      "Normalize case before canonical set join (§Standardization). "
                       "Register one canonical form; register case variants as aliases."},
    ],
}

print(f"Constants loaded: {len(TAGS)} tags | {len(STANDARDIZATION_RULES)} standardization entries")
print(f"Categorical domains: {len(CATEGORICAL_DOMAINS)}")


In [0]:
# ---------------------------------------------------------------------------
# DISCOVERY:
# Scans information_schema for STRING columns that are candidates for
# categorical analysis. Uses column name heuristics as a strong prior.
# Also loads the canonical lookup table if configured (Method B).
# ---------------------------------------------------------------------------

cat, sch = CFG["catalog"], CFG["schema"]

def _esc(name: str) -> str:
    return name.replace("`", "``")

STRING_DTYPES_PFX = ("STRING", "VARCHAR", "CHAR")

view_clause = "AND table_type IN ('MANAGED', 'EXTERNAL')" if CFG["skip_views"] else ""
tables: List[str] = [
    r.table_name for r in spark.sql(f"""
        SELECT table_name FROM `{_esc(cat)}`.information_schema.tables
        WHERE  table_schema = '{_esc(sch)}' {view_clause}
        ORDER BY table_name
    """).collect()
]

if CFG["table_filter"].strip():
    pat = re.compile(CFG["table_filter"], re.I)
    tables = [t for t in tables if pat.search(t)]

if not tables:
    raise ValueError(f"No tables found in {cat}.{sch} -- check catalog/schema/table_filter.")

tbl_in = ", ".join(f"'{_esc(t)}'" for t in tables)
table_cols: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM   `{_esc(cat)}`.information_schema.columns
    WHERE  table_schema = '{_esc(sch)}' AND table_name IN ({tbl_in})
    ORDER BY table_name, ordinal_position
""").collect():
    dt = r.data_type.upper()
    if any(dt.startswith(p) for p in STRING_DTYPES_PFX):
        table_cols[r.table_name].append((r.column_name, dt))

# ── Load canonical lookup table (Method B) ───────────────────────────────────
# Structure: domain STRING, canonical_value STRING
# domain is matched against column name as a substring (case-insensitive)
CANONICAL_LOOKUP: Dict[str, Set[str]] = {}  # domain -> set of canonical values (uppercased)

cc, cs, ct = CFG["canonical_catalog"], CFG["canonical_schema"], CFG["canonical_table"]
if cc and cs and ct:
    try:
        lookup_df = spark.sql(f"""
            SELECT LOWER(TRIM(domain)) AS domain, UPPER(TRIM(canonical_value)) AS canonical_value
            FROM `{_esc(cc)}`.`{_esc(cs)}`.`{_esc(ct)}`
            WHERE domain IS NOT NULL AND canonical_value IS NOT NULL
        """)
        for r in lookup_df.collect():
            CANONICAL_LOOKUP.setdefault(r["domain"], set()).add(r["canonical_value"])
        print(f"Canonical lookup loaded: {len(CANONICAL_LOOKUP)} domain(s), "
              f"{sum(len(v) for v in CANONICAL_LOOKUP.values())} total values")
    except Exception as e:
        print(f"  [WARN] Could not load canonical lookup table: {e}")
else:
    print("Canonical lookup table: not configured (using inline sets only)")

# Merge inline + lookup canonical sets
# get_canonical_set(col) returns the set of canonical values for a column, or None
def get_canonical_set(col: str) -> Optional[Set[str]]:
    cl = col.lower()
    # Method B: lookup table match (substring)
    for domain, values in CANONICAL_LOOKUP.items():
        if domain in cl:
            return values
    # Method A: inline match (substring)
    for key, values in CANONICAL_INLINE.items():
        if key.lower() in cl:
            return values
    return None

total_string_cols = sum(len(v) for v in table_cols.values())
print(f"Scope        : {cat}.{sch}")
print(f"Tables       : {len(tables)}")
print(f"String cols  : {total_string_cols}")
print(f"Layer        : {CFG['layer']}")


In [0]:
# ---------------------------------------------------------------------------
# PROFILER -- key design decisions:
#
# 1. TWO-PASS approach per table:
#    Pass 1 (cheap): ONE SQL for null_count, total, approx_count_distinct,
#                    avg_length for ALL STRING columns at once.
#                    Use approx_count_distinct (fast, scalable) to identify
#                    low-cardinality candidates.
#    Pass 2 (targeted): For confirmed candidates only, ONE SQL gets exact
#                    distinct count and the full distinct value list.
#                    This keeps cost proportional to the number of categorical
#                    columns, not the total column count.
#
# 2. Distinct value list is the key output -- it feeds both the AI classifier
#    and the canonical set compliance check in the Rule Engine.
#
# 3. Mixed-case detection: for each column check if the same value appears
#    in multiple case variants (male/Male/MALE) using the distinct value list.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")

def get_sample(tbl: str) -> DataFrame:
    fq = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    n  = CFG["sample_size"]
    try:
        return spark.sql(f"SELECT * FROM {fq} TABLESAMPLE ({n} ROWS)")
    except Exception:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fq}").collect()[0]["n"]
        return (
            spark.table(fq)
            .sample(withReplacement=False, fraction=min(1.0, n / max(1, total)), seed=CFG["seed"])
            .limit(n)
        )


def pass1_screen(tbl: str, cols: List[Tuple[str, str]]) -> List[Tuple[str, str, int, int, float]]:
    """
    Pass 1: ONE SQL per table. Returns list of (col, dtype, approx_distinct, null_count, avg_len)
    for ALL STRING columns. Uses APPROX_COUNT_DISTINCT for speed at scale.
    """
    fq    = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    exprs = ["COUNT(*) AS __total__"]
    for col, dtype in cols:
        cs   = f"`{_esc(col)}`"
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        exprs += [
            f"COUNT(*) - COUNT({cs}) AS `__null__{safe}`",
            f"APPROX_COUNT_DISTINCT({cs}) AS `__acd__{safe}`",
            f"COALESCE(AVG(CASE WHEN {cs} IS NOT NULL THEN LENGTH(TRIM({cs})) END), 0) "
            f"AS `__avglen__{safe}`",
        ]
    try:
        row   = spark.sql(f"SELECT {', '.join(exprs)} FROM {fq}").collect()[0].asDict()
        total = row["__total__"] or 0
        results = []
        for col, dtype in cols:
            safe       = col.replace("`","").replace(" ","_").replace(".","_")[:60]
            null_count = int(row.get(f"__null__{safe}", 0) or 0)
            acd        = int(row.get(f"__acd__{safe}", 0) or 0)
            avg_len    = float(row.get(f"__avglen__{safe}", 0) or 0)
            results.append((col, dtype, acd, null_count, avg_len, total))
        return results
    except Exception as e:
        print(f"  [WARN] Pass 1 failed for {tbl}: {e}")
        return []


def is_categorical_candidate(col: str, acd: int, total: int, avg_len: float) -> bool:
    """
    A column is a categorical candidate if:
    - name matches RE_CATEGORICAL_NAME, OR low cardinality ratio
    - AND not excluded by RE_CATEGORICAL_EXCLUDE
    - AND avg_length <= max_avg_length (excludes narratives/free-text)
    - AND total >= min_rows
    """
    if RE_CATEGORICAL_EXCLUDE.search(col):
        return False
    if total < CFG["min_rows"]:
        return False
    if avg_len > CFG["max_avg_length"]:
        return False
    name_match = bool(RE_CATEGORICAL_NAME.search(col))
    ratio_ok   = total > 0 and (acd / total) <= CFG["cardinality_ratio"] and acd <= CFG["max_cardinality"]
    return name_match or ratio_ok


def pass2_profile(tbl: str, candidates: List[Tuple[str, str]]) -> Dict[str, dict]:
    """
    Pass 2: ONE SQL per table for confirmed candidates only.
    Gets exact distinct count and full distinct value list per column.
    Distinct values are collected via GROUP BY (exact, not approximate).
    Max 200 distinct values collected per column (enough for compliance checking).
    """
    if not candidates:
        return {}

    fq      = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    results = {}

    for col, dtype in candidates:
        cs   = f"`{_esc(col)}`"
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        try:
            # Collect distinct values with counts (top 200 by frequency)
            dist_rows = spark.sql(f"""
                SELECT TRIM({cs}) AS val, COUNT(*) AS cnt
                FROM `{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`
                WHERE {cs} IS NOT NULL AND TRIM({cs}) != ''
                GROUP BY TRIM({cs})
                ORDER BY cnt DESC
                LIMIT 200
            """).collect()

            distinct_vals    = {r["val"]: int(r["cnt"]) for r in dist_rows}
            exact_distinct   = len(distinct_vals)
            # Check for mixed-case: same value in multiple cases
            lower_vals       = [v.lower() for v in distinct_vals]
            mixed_case_groups= []
            seen_lower       = {}
            for v in distinct_vals:
                lv = v.lower()
                seen_lower.setdefault(lv, []).append(v)
            mixed_case_groups = [variants for variants in seen_lower.values() if len(variants) > 1]

            results[col] = {
                "dtype":              dtype,
                "distinct_vals":      distinct_vals,   # {value: count}
                "exact_distinct":     exact_distinct,
                "mixed_case_groups":  mixed_case_groups,
                "canonical_set":      get_canonical_set(col),  # None if not registered
            }
        except Exception as e:
            print(f"  [WARN] Pass 2 failed for {tbl}.{col}: {e}")

    return results


# ── Run profiler ──────────────────────────────────────────────────────────────
table_samples:    Dict[str, DataFrame]       = {}
cat_candidates:   Dict[str, Dict[str, dict]] = {}  # tbl -> {col -> profile}
total_screened    = 0
total_candidates  = 0

for tbl in tables:
    cols = table_cols.get(tbl, [])
    if not cols:
        continue

    # Pass 1: cheap screening
    p1 = pass1_screen(tbl, cols)
    total_screened += len(cols)

    # Filter to categorical candidates
    confirmed_cols = [
        (col, dtype)
        for col, dtype, acd, null_count, avg_len, total in p1
        if is_categorical_candidate(col, acd, total, avg_len)
    ]

    # Store null/total from pass1 alongside pass2 results
    p1_stats = {col: {"null_count": null_count, "total": total, "approx_distinct": acd}
                for col, dtype, acd, null_count, avg_len, total in p1}

    if confirmed_cols:
        # Pass 2: exact distinct values
        p2 = pass2_profile(tbl, confirmed_cols)
        # Merge pass1 stats into pass2 results
        for col, profile in p2.items():
            profile.update(p1_stats.get(col, {}))
        cat_candidates[tbl] = p2
        total_candidates += len(p2)

print(f"Profiler done.")
print(f"  Screened  : {total_screened} string columns across {len(tables)} table(s)")
print(f"  Candidates: {total_candidates} categorical column(s)")
for tbl, cols in cat_candidates.items():
    print(f"  {tbl}: {len(cols)} candidate(s): {list(cols.keys())}")


In [0]:
# ---------------------------------------------------------------------------
# AI CLASSIFIER -- chunked ai_query() calls (BATCH_SIZE=20).
#
# Three responsibilities:
# 1. COLUMN CONFIRMATION: Is this genuinely a categorical field?
#    (Rejects: free-text disguised as low-cardinality due to small sample,
#     encoded IDs with few active values, boolean-looking but not categorical)
# 2. DOMAIN CLASSIFICATION: What categorical domain does this belong to?
#    (gender_sex, race_ethnicity, claim_status, yes_no_flag, etc.)
# 3. ALIAS DETECTION (compliance mode only): For columns with a canonical set,
#    identify which out-of-set values are likely aliases of canonical values
#    vs. genuinely new/invalid values.
# ---------------------------------------------------------------------------

BATCH_SIZE = 20

def _ai_query(prompt: str) -> str:
    safe = prompt.replace("'", "''")
    raw  = spark.sql(
        f"SELECT ai_query('{CFG['ai_model']}', '{safe}', failOnError => false) AS r"
    ).collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)

def _chunked(items: list, size: int = BATCH_SIZE):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def classify_table(tbl: str) -> None:
    cands = cat_candidates.get(tbl, {})
    if not cands:
        return

    if not CFG["enable_ai"]:
        for col, info in cands.items():
            info["is_categorical"]  = True
            info["domain"]          = "other"
            info["ai_confidence"]   = "low"
            info["alias_map"]       = {}
        return

    col_list = list(cands.keys())

    # ── Step 1: Confirm + classify domain ─────────────────────────────────────
    for chunk in _chunked(col_list):
        payload = json.dumps([{
            "col":            col,
            "table":          tbl,
            "distinct_count": cands[col].get("exact_distinct", 0),
            "total_rows":     cands[col].get("total", 0),
            # Top 20 most frequent values (sorted by count desc)
            "top_values":     list(cands[col].get("distinct_vals", {}).keys())[:20],
            "approx_distinct":cands[col].get("approx_distinct", 0),
        } for col in chunk if col in cands], default=str)

        prompt = (
            "Healthcare data governance expert. Reply ONLY with a JSON array -- no prose, no markdown. "
            f"Table: {tbl}. For each STRING column determine: "
            "(1) is_categorical: true if this column stores a fixed set of category/status/type codes "
            "or labels. False if: free-text (even short), encoded IDs with few active values, "
            "boolean-only columns that should be BOOLEAN type, date/time values, or numeric codes. "
            "(2) domain: pick one from: gender_sex, race_ethnicity, claim_status, enrollment_status, "
            "coverage_type, diagnosis_category, service_type, provider_type, address_type, "
            "contact_type, language, yes_no_flag, payer_type, relationship, encounter_type, "
            "discharge_status, other. "
            "(3) confidence: high/medium/low. "
            f"Columns: {payload}. "
            'Return: [{"col":"<n>","is_categorical":true|false,"domain":"<d>","confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                col = item.get("col", "")
                if col not in cands:
                    continue
                cands[col].update({
                    "is_categorical": item.get("is_categorical", True),
                    "domain":         item.get("domain", "other"),
                    "ai_confidence":  item.get("confidence", "low"),
                    "alias_map":      {},
                })
        except Exception as e:
            print(f"  [WARN] AI classification chunk failed for {tbl}: {e}")
            for col in chunk:
                if col in cands:
                    cands[col].update({
                        "is_categorical": True,  # conservative
                        "domain":         "other",
                        "ai_confidence":  "low",
                        "alias_map":      {},
                        "ai_error":       str(e),
                    })

    # Remove non-categorical columns
    for col in list(cat_candidates[tbl].keys()):
        if not cat_candidates[tbl][col].get("is_categorical", True):
            del cat_candidates[tbl][col]

    if not cat_candidates[tbl]:
        return

    # ── Step 2: Alias detection (compliance mode only) ─────────────────────────
    # For columns WITH a canonical set, find which out-of-set values look like aliases
    compliance_cols = [
        col for col in cat_candidates[tbl]
        if cat_candidates[tbl][col].get("canonical_set") is not None
    ]

    for chunk in _chunked(compliance_cols):
        chunk_data = []
        for col in chunk:
            if col not in cat_candidates[tbl]:
                continue
            info         = cat_candidates[tbl][col]
            canonical    = info.get("canonical_set", set())
            # Compute out-of-set distinct values for alias analysis
            oos_vals     = [
                v for v in info.get("distinct_vals", {})
                if v.upper() not in canonical and v.strip()
            ][:30]  # cap at 30 for prompt size
            if not oos_vals:
                continue
            chunk_data.append({
                "col":            col,
                "domain":         info.get("domain", "other"),
                "canonical_set":  sorted(canonical)[:30],
                "oos_values":     oos_vals,
            })

        if not chunk_data:
            continue

        payload = json.dumps(chunk_data, default=str)
        prompt = (
            "Healthcare data governance expert. Reply ONLY with a JSON array -- no prose, no markdown. "
            "For each column, identify which out-of-set values are likely ALIASES of canonical values "
            "(e.g. Male→M, Female→F, Active→ACTIVE, Yes→Y) vs. genuinely new/invalid values. "
            "An alias is a different representation of the same logical concept. "
            "A new/invalid value is a genuinely different concept or garbage. "
            f"Columns: {payload}. "
            'Return: [{"col":"<n>","alias_map":{"<oos_value>":"<canonical_value>"}}] '
            "-- only include values you are confident are aliases. Omit genuinely new values."
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                col = item.get("col", "")
                if col not in cat_candidates[tbl]:
                    continue
                cat_candidates[tbl][col]["alias_map"] = item.get("alias_map", {})
        except Exception as e:
            print(f"  [WARN] AI alias detection failed for {tbl}: {e}")


# ── Run classifier ────────────────────────────────────────────────────────────
for tbl in list(cat_candidates.keys()):
    classify_table(tbl)
    remaining = len(cat_candidates[tbl])
    domains = {}
    for info in cat_candidates[tbl].values():
        d = info.get("domain", "other")
        domains[d] = domains.get(d, 0) + 1
    dom_str = ", ".join(f"{d}:{n}" for d, n in sorted(domains.items()))
    print(f"  ok {tbl}: {remaining} categorical column(s) [{dom_str}]")

total_confirmed = sum(len(v) for v in cat_candidates.values())
print(f"\nAI classification done. {total_confirmed} confirmed categorical column(s).")


In [0]:
# ---------------------------------------------------------------------------
# RULE ENGINE:
# All stats come from pre-computed cat_candidates dict.
# No additional full-table SQL runs here.
#
# Two operating modes within the same function:
#   INVENTORY MODE: canonical_set is None → fire UNREGISTERED_CATEGORICAL
#                   (and RAW_VALUES_IN_GOLD for Gold layer)
#   COMPLIANCE MODE: canonical_set is a Set → fire OOS, HIGH_OOS, UNMAPPED,
#                    UNDOCUMENTED_ALIAS, CARDINALITY_DRIFT
# Both modes always check MIXED_CASE_CATEGORICAL and HIGH_NULL_RATE.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")


def _finding(tbl, col, tag, count, total, samples, action,
             hint=None, confidence="high",
             std_options=None, auto_decided_action=None,
             domain="other", canonical_set_size=0) -> dict:
    pct     = round(count / total * 100, 4) if total else 0.0
    options = std_options if std_options is not None else STANDARDIZATION_RULES.get(tag, [])
    decided = auto_decided_action if auto_decided_action is not None else None
    return {
        "run_id":                   RUN_ID,
        "run_ts":                   RUN_TS,
        "catalog":                  cat,
        "schema":                   sch,
        "table_name":               tbl,
        "column_name":              col,
        "domain":                   domain,
        "layer":                    CFG["layer"],
        "rule_ref":                 TAGS.get(tag, "§?"),
        "classification":           tag,
        "canonical_set_size":       canonical_set_size,
        "affected_count":           int(count),
        "affected_pct":             float(pct),
        "total_rows":               int(total),
        "sample_values":            json.dumps(samples, default=str),
        "recommended_action":       action,
        "dictionary_strategy_hint": hint,
        "standardization_rule":     json.dumps(options, ensure_ascii=False),
        "confidence":               confidence,
        "needs_steward_review":     decided is None,
        "decided_action":           decided,
        "decided_by":               None,
    }


def _check_categorical(tbl: str, col: str, info: dict) -> list:
    total          = info.get("total", 0)
    null_count     = info.get("null_count", 0)
    non_null       = total - null_count
    distinct_vals  = info.get("distinct_vals", {})   # {value: count}
    exact_distinct = info.get("exact_distinct", 0)
    canonical_set  = info.get("canonical_set")        # None = inventory mode
    alias_map      = info.get("alias_map", {})        # {oos_val: canonical_val}
    domain         = info.get("domain", "other")
    confidence     = info.get("ai_confidence", "low")
    mixed_groups   = info.get("mixed_case_groups", [])
    canon_size     = len(canonical_set) if canonical_set else 0
    findings       = []

    # Sample values for context (top 10 most frequent)
    sample_vals = list(distinct_vals.keys())[:10]

    # ── §Validation: HIGH_NULL_RATE ───────────────────────────────────────────
    if total > 0:
        null_pct = null_count / total * 100
        if null_pct > CFG["null_rate_threshold"]:
            findings.append(_finding(tbl, col, "HIGH_NULL_RATE",
                null_count, total, [],
                f"{null_pct:.1f}% of values are NULL in categorical column '{col}'. "
                "May indicate upstream mapping failure or data loss (§Validation).",
                confidence="medium",
                domain=domain, canonical_set_size=canon_size,
            ))

    # ── §Standardization: MIXED_CASE_CATEGORICAL ──────────────────────────────
    if mixed_groups:
        all_variants = [v for grp in mixed_groups for v in grp]
        findings.append(_finding(tbl, col, "MIXED_CASE_CATEGORICAL",
            len(all_variants), total, all_variants[:10],
            f"Column '{col}' contains {len(mixed_groups)} value(s) that appear in "
            "multiple case variants "
            f"(e.g. {mixed_groups[0]}). "
            "Normalize case to one canonical form in Silver ETL (§Standardization). "
            "Register case variants as aliases in the alias table.",
            confidence="high",
            domain=domain, canonical_set_size=canon_size,
        ))

    # ── INVENTORY MODE (no canonical set registered) ──────────────────────────
    if canonical_set is None:

        # §Inventory: UNREGISTERED_CATEGORICAL
        findings.append(_finding(tbl, col, "UNREGISTERED_CATEGORICAL",
            exact_distinct, total, sample_vals,
            f"Column '{col}' appears categorical (domain: {domain}, "
            f"{exact_distinct} distinct value(s)) but has no registered canonical value set. "
            "Catalog in data dictionary with expected cardinality and steward owner (§Inventory). "
            "Register canonical set via lookup table (preferred) or inline mapping (§Standardization).",
            confidence=confidence,
            domain=domain, canonical_set_size=0,
        ))

        # §Inventory: RAW_VALUES_IN_GOLD (Gold/certified layer only)
        if CFG["layer"] in ("Gold",):
            findings.append(_finding(tbl, col, "RAW_VALUES_IN_GOLD",
                non_null, total, sample_vals,
                f"Column '{col}' in Gold layer has {non_null:,} non-null value(s) with "
                "no registered canonical mapping. "
                "Raw source strings must not reach Gold/certified models when a canonical set exists (§Standardization). "
                "Register canonical set and apply ETL mapping. Certification blocker (§Enforcement).",
                confidence="high",
                domain=domain, canonical_set_size=0,
            ))

        return findings

    # ── COMPLIANCE MODE (canonical set registered) ────────────────────────────

    # Normalize distinct values for comparison (uppercase, trimmed)
    oos_vals  = {}   # {original_value: count} where UPPER(value) NOT IN canonical_set
    in_set_count = 0

    for val, cnt in distinct_vals.items():
        normalized = val.upper().strip()
        if normalized in canonical_set:
            in_set_count += cnt
        else:
            oos_vals[val] = cnt

    oos_count = sum(oos_vals.values())
    oos_distinct_count = len(oos_vals)

    # §Validation: OUT_OF_SET_VALUE (unconditional -- any OOS value fires)
    if oos_count > 0:
        oos_samples = list(oos_vals.keys())[:10]
        findings.append(_finding(tbl, col, "OUT_OF_SET_VALUE",
            oos_count, total, oos_samples,
            f"{oos_count:,} value(s) ({oos_count/total*100:.1f}%) in '{col}' are not in "
            f"the approved canonical set ({canon_size} values). "
            "DQ rules must fail or flag rows where post-transform value not in approved set (§Validation). "
            "NULL or quarantine out-of-set values; do not pass raw strings to Gold (§Standardization).",
            confidence="high",
            domain=domain, canonical_set_size=canon_size,
        ))

    # §Validation: HIGH_OUT_OF_SET_RATE (rate-threshold)
    if non_null > 0 and oos_count > 0:
        oos_pct = oos_count / non_null * 100
        if oos_pct > CFG["high_oos_threshold"]:
            findings.append(_finding(tbl, col, "HIGH_OUT_OF_SET_RATE",
                oos_count, total, list(oos_vals.keys())[:5],
                f"{oos_pct:.1f}% of non-null values in '{col}' are out-of-set. "
                "Uncertified category drift in Gold layers blocks certification (§Enforcement). "
                "Monitor alias table for unmapped values weekly during onboarding, "
                "daily for production-critical domains (§Validation).",
                confidence="high",
                auto_decided_action="escalate_and_quarantine",
                domain=domain, canonical_set_size=canon_size,
            ))

    # §Validation: UNMAPPED_VALUES (inventory of OOS distinct values)
    if oos_distinct_count > 0:
        # Separate: AI-detected aliases vs. genuinely unmapped
        alias_vals   = {v: c for v, c in oos_vals.items() if v in alias_map}
        unmapped_vals= {v: c for v, c in oos_vals.items() if v not in alias_map}

        if unmapped_vals:
            findings.append(_finding(tbl, col, "UNMAPPED_VALUES",
                len(unmapped_vals), total,
                list(unmapped_vals.keys())[:20],
                f"{len(unmapped_vals)} distinct value(s) in '{col}' have no mapping to the "
                f"canonical set. Feed into alias table review (§Change-control). "
                "Adding aliases requires PR/review; new canonical categories require "
                "product owner sign-off (§Change-control).",
                confidence="high",
                domain=domain, canonical_set_size=canon_size,
            ))

        # §Validation: UNDOCUMENTED_ALIAS (AI-detected aliases)
        if alias_vals:
            alias_examples = [f"{v}→{alias_map[v]}" for v in list(alias_vals.keys())[:5]]
            findings.append(_finding(tbl, col, "UNDOCUMENTED_ALIAS",
                sum(alias_vals.values()), total,
                alias_examples,
                f"{len(alias_vals)} value(s) in '{col}' appear to be aliases of canonical values "
                f"(e.g. {alias_examples[:2]}). "
                "Register in alias table via PR/review (§Change-control). "
                "Normalize to canonical form in Silver ETL after alias is approved.",
                confidence="medium",
                domain=domain, canonical_set_size=canon_size,
            ))

    # §Change control: CARDINALITY_DRIFT
    # Check against expected cardinality if configured
    expected = None
    for key, val in EXPECTED_CARDINALITY.items():
        if key.lower() in col.lower():
            expected = val
            break
    if expected is not None and exact_distinct > expected:
        new_vals = [v for v in distinct_vals if v.upper() not in canonical_set][:10]
        findings.append(_finding(tbl, col, "CARDINALITY_DRIFT",
            exact_distinct - expected, total, new_vals,
            f"Column '{col}' has {exact_distinct} distinct value(s), exceeding expected "
            f"cardinality of {expected}. "
            "New values may require product owner sign-off (§Change-control). "
            "Do not add new canonical values without steward approval and dictionary update.",
            confidence="medium",
            domain=domain, canonical_set_size=canon_size,
        ))

    return findings


# ── Main loop ─────────────────────────────────────────────────────────────────
all_findings: List[dict] = []

for tbl, cols in cat_candidates.items():
    tbl_count = 0
    for col, info in cols.items():
        col_findings = _check_categorical(tbl, col, info)
        all_findings.extend(col_findings)
        tbl_count += len(col_findings)
    print(f"  ok {tbl}: {tbl_count} finding(s)")

print(f"\nRule engine done. {len(all_findings)} total finding(s).")


In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType, IntegerType)

SCHEMA = StructType([
    StructField("run_id",                   StringType(),   True),
    StructField("run_ts",                   StringType(),   True),
    StructField("catalog",                  StringType(),   True),
    StructField("schema",                   StringType(),   True),
    StructField("table_name",               StringType(),   True),
    StructField("column_name",              StringType(),   True),
    StructField("domain",                   StringType(),   True),
    StructField("layer",                    StringType(),   True),
    StructField("rule_ref",                 StringType(),   True),
    StructField("classification",           StringType(),   True),
    StructField("canonical_set_size",       IntegerType(),  True),
    StructField("affected_count",           LongType(),     True),
    StructField("affected_pct",             DoubleType(),   True),
    StructField("total_rows",               LongType(),     True),
    StructField("sample_values",            StringType(),   True),
    StructField("recommended_action",       StringType(),   True),
    StructField("dictionary_strategy_hint", StringType(),   True),
    StructField("standardization_rule",     StringType(),   True),
    StructField("confidence",               StringType(),   True),
    StructField("needs_steward_review",     BooleanType(),  True),
    StructField("decided_action",           StringType(),   True),
    StructField("decided_by",               StringType(),   True),
])

resolved_out_catalog = CFG.get('out_catalog') or CFG['catalog']
resolved_out_schema  = CFG.get('out_schema') or CFG['schema']

out_fq  = f"`{resolved_out_catalog}`.`{resolved_out_schema}`.`{CFG['out_table']}`"
out_tbl = f"{resolved_out_catalog}.{resolved_out_schema}.{CFG['out_table']}"

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{resolved_out_catalog}`.`{resolved_out_schema}`")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_tbl)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No findings generated.")

print(f"Run ID: {RUN_ID}")


In [0]:
BLOCKING = {
    "RAW_VALUES_IN_GOLD", "OUT_OF_SET_VALUE", "HIGH_OUT_OF_SET_RATE",
}

if not all_findings:
    print("No categorical findings. Run complete.")
else:
    fdf = findings_df

    # Section 1 -- Blocking (certification blockers)
    block_df = fdf.filter(F.col("classification").isin(BLOCKING)) \
                  .orderBy(F.col("affected_pct").desc())
    n_block = block_df.count()
    print("=" * 70)
    print(f"  SECTION 1 -- BLOCKING FINDINGS (certification blockers): {n_block}")
    print("=" * 70)
    if n_block:
        display(block_df.select(
            "table_name","column_name","domain","layer","classification",
            "canonical_set_size","affected_count","affected_pct",
            "sample_values","recommended_action","decided_action","decided_by"
        ))
    else:
        print("  None.")

    # Section 2 -- By classification
    print("\n" + "-" * 70)
    print("SECTION 2 -- Findings by classification")
    print("-" * 70)
    display(
        fdf.groupBy("classification","rule_ref","domain")
           .agg(
               F.count("*").alias("findings"),
               F.countDistinct("table_name").alias("tables"),
               F.countDistinct("column_name").alias("columns"),
               F.sum("affected_count").alias("total_affected"),
           ).orderBy(F.col("findings").desc())
    )

    # Section 3 -- Inventory mode (unregistered columns)
    inv_df = fdf.filter(F.col("classification").isin(
        "UNREGISTERED_CATEGORICAL", "RAW_VALUES_IN_GOLD"
    ))
    n_inv = inv_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 3 -- Inventory: unregistered categorical columns ({n_inv})")
    print("  These columns need canonical set registration before compliance can be measured")
    print("-" * 70)
    if n_inv:
        display(inv_df.select(
            "table_name","column_name","domain","layer","classification",
            "affected_count","sample_values","recommended_action","decided_action","decided_by"
        ).orderBy("classification","table_name","column_name"))
    else:
        print("  None -- all categorical columns have registered canonical sets.")

    # Section 4 -- Unmapped values (alias table feed)
    alias_df = fdf.filter(F.col("classification").isin(
        "UNMAPPED_VALUES", "UNDOCUMENTED_ALIAS"
    ))
    n_alias = alias_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 4 -- Unmapped values and aliases ({n_alias})")
    print("  These need alias table review; adding aliases requires PR/review (§Change-control)")
    print("-" * 70)
    if n_alias:
        display(alias_df.select(
            "table_name","column_name","domain","classification",
            "affected_count","sample_values","recommended_action","decided_action","decided_by"
        ).orderBy("classification","table_name","column_name"))
    else:
        print("  None.")

    # Section 5 -- Cardinality drift and new values
    drift_df = fdf.filter(F.col("classification").isin(
        "CARDINALITY_DRIFT", "NEW_VALUE_APPEARED"
    ))
    n_drift = drift_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 5 -- Cardinality drift and new values ({n_drift})")
    print("  New canonical categories require product owner sign-off (§Change-control)")
    print("-" * 70)
    if n_drift:
        display(drift_df.select(
            "table_name","column_name","domain","canonical_set_size",
            "classification","affected_count","sample_values",
            "recommended_action","decided_action","decided_by"
        ).orderBy("table_name","column_name"))
    else:
        print("  None.")

    # Section 6 -- Normalization issues
    norm_df = fdf.filter(F.col("classification").isin(
        "MIXED_CASE_CATEGORICAL", "HIGH_NULL_RATE", "INLINE_MAPPING_RISK"
    ))
    n_norm = norm_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 6 -- Normalization issues ({n_norm})")
    print("-" * 70)
    if n_norm:
        display(norm_df.select(
            "table_name","column_name","domain","classification",
            "affected_count","affected_pct","sample_values",
            "recommended_action","decided_action","decided_by"
        ).orderBy("table_name","column_name"))
    else:
        print("  None.")

    # Section 7 -- Engineer work queue
    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 7 -- Engineer work queue ({n_work} decided)")
    print("-" * 70)
    if n_work:
        display(work_df.select(
            "table_name","column_name","domain","classification",
            "decided_action","decided_by","standardization_rule"
        ).orderBy("table_name","column_name"))
    else:
        print("  No decisions recorded yet.")
        print(f"  Query: SELECT * FROM {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")
        print(f"  WHERE run_id = '{RUN_ID}' AND decided_action IS NULL")

    # Distinct value inventory for steward (inventory mode tables)
    print("\n" + "-" * 70)
    print("SECTION 8 -- Distinct value inventory (all confirmed categorical columns)")
    print("  Use this to seed canonical set registration")
    print("-" * 70)
    inventory = []
    for tbl, cols in cat_candidates.items():
        for col, info in cols.items():
            for val, cnt in list(info.get("distinct_vals", {}).items())[:50]:
                canonical  = info.get("canonical_set")
                is_in_set  = canonical is not None and val.upper().strip() in canonical
                is_alias   = val in info.get("alias_map", {})
                inventory.append({
                    "table_name":     tbl,
                    "column_name":    col,
                    "domain":         info.get("domain", "other"),
                    "distinct_value": val,
                    "row_count":      cnt,
                    "in_canonical_set": is_in_set,
                    "is_alias":       is_alias,
                    "canonical_set_size": len(canonical) if canonical else 0,
                })
    if inventory:
        inv_sdf = spark.createDataFrame(inventory)
        display(inv_sdf.orderBy("table_name","column_name","row_count", ascending=[True,True,False]))

    print("\n" + "=" * 70)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  |  Layer: {CFG['layer']}")
    print(f"  Categorical columns confirmed: {total_confirmed}")
    print(f"  Findings: {len(all_findings):,}  |  Blocking: {n_block}  |  Unregistered: {n_inv}  |  Unmapped: {n_alias}  |  Drift: {n_drift}")
    print("=" * 70)
    print("\nDetection-only. No source data was modified.")
    print("Every finding requires data steward review before any action.")
